In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

D:\agentic_ai_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [3]:
llm=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message= HumanMessage(
    content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text", "text": "This is a blood work report.Extract all test resulte and flag any value outside the normal range."}
    ]
)
response = llm.invoke([message])
print(response.content)

Here are the test results extracted from the blood work report with values outside the normal range flagged:

**COMPLETE BLOOD COUNT (CBC)**

* Hemoglobin: 15.1 g/dL (Normal: 13.5-17.5) **(Within Normal Range)**
* Hematocrit: 44% (Normal: 41-53) **(Within Normal Range)**
* WBC: 6.8 x10^3/uL (Normal: 4.5-11.0) **(Within Normal Range)**
* Platelets: 220 x10^3/uL (Normal: 150-400) **(Within Normal Range)**

**LIPID PANEL**

* Total Cholesterol: 238 mg/dL (Normal: <200) **(Elevated)**
* LDL Cholesterol: 162 mg/dL (Normal: <100) **(Elevated)**
* HDL Cholesterol: 36 mg/dL (Normal: >40) **(Low)**
* Triglycerides: 188 mg/dL (Normal: <150) **(Elevated)**

**METABOLIC PANEL**

* Glucose (Fasting): 92 mg/dL (Normal: 70-99) **(Within Normal Range)**
* HbA1c: 5.3% (Normal: <5.7) **(Within Normal Range)**
* Creatinine: 1.0 mg/dL (Normal: 0.7-1.3) **(Within Normal Range)**
* eGFR: 82 mL/min (Normal: >60) **(Within Normal Range)**

The values outside the normal range are:

* Elevated:
	+ Total Cholest

In [7]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Give a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_suger."""

    diet_plans = {
        "high_cholesterol" : {
            "eat" : ["fruits","vegitables", "whole grains" , "lean protein"],
            "do_not_eat" : ["read meat","fried food","full fat dairy", "processed snacks"],
        },
        "high_suger" : {
            "eat" : ["vegitables", "whole grains", "legumes", "nuts"],
            "do_not_eat" : ["white rice", "white suger", "junk food", "sugary drinks"],
        },
        "normal" : {
            "eat" : ["vegetables", "fruits", "white grains", "lean protien"],
            "do_not_eat" : ["excessive suger","processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [8]:
SYSTEM_PROMPT = """
You are a helpfull medical and nutrition assistant
for the blood work input image, extract the numbers and the normal range, then categorize 
the condition as one of: normal, high_cholesterol, high_suger.
Then call the appropriate tool to retrieve and present the diet plan."""

diet_agent = create_agent(
    llm,
    tools = [get_diet_recommendation],
    system_prompt = SYSTEM_PROMPT,
)

In [10]:
result = diet_agent.invoke({
    "message": [HumanMessage(content=[
        {"type": "image_url","image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"typr": "text", "text": "Analyse this blood work report and suggest a diet plan."}
    ])]
})

print(result["messages"][-1].content)

NameError: name 'diet_plans' is not defined